### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [2]:
%pip install --upgrade transformers datasets accelerate deepspeed -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Load data and model

In [3]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
qqp

DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 390965
    })
})

In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: gchhablani/bert-base-cased-finetuned-qqp
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Tokenize the data

In [ ]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [ ]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=32, shuffle=False, num_workers=2, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
    print(batch)
    break

{'labels': tensor([0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0]), 'idx': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]), 'input_ids': tensor([[ 101, 2009, 1132,  ...,    0,    0,    0],
        [ 101,  146, 1328,  ...,    0,    0,    0],
        [ 101, 2181, 1175,  ...,    0,    0,    0],
        ...,
        [ 101,  107, 2627,  ...,    0,    0,    0],
        [ 101, 1327, 2146,  ...,    0,    0,    0],
        [ 101, 2009, 1674,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 

In [ ]:
correct = 0
total = 0

with torch.no_grad():
  for i, batch in enumerate(val_loader):
      outputs = model(
          input_ids=batch['input_ids'].to(device),
          attention_mask=batch['attention_mask'].to(device),
          token_type_ids=batch['token_type_ids'].to(device)
      )

      answer_model = outputs.logits.argmax(dim=-1)
      labels = batch['labels'].to(device)

      correct += (answer_model == labels).sum().item()
      total += labels.size(0)

      if i % 100 == 0:
          print(f'Batch {i}/{len(val_loader)} is DONE!')

Batch 0/1264 is DONE!
Batch 100/1264 is DONE!
Batch 200/1264 is DONE!
Batch 300/1264 is DONE!
Batch 400/1264 is DONE!
Batch 500/1264 is DONE!
Batch 600/1264 is DONE!
Batch 700/1264 is DONE!
Batch 800/1264 is DONE!
Batch 900/1264 is DONE!
Batch 1000/1264 is DONE!
Batch 1100/1264 is DONE!
Batch 1200/1264 is DONE!


__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
accuracy = correct / total
print(accuracy)

0.9083848627256987


In [ ]:
assert 0.9 < accuracy < 0.91

In [ ]:
# вынесем в отдельную функцию вычисление метрики
def evaluate(model, val_loader, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
      for i, batch in enumerate(val_loader):
          batch = {k: v.to(device) for k, v in batch.items()}
          outputs = model(**batch)

          answer_model = outputs.logits.argmax(dim=-1)
          loss = outputs.loss
          labels = batch['labels'].to(device)

          total_loss += loss.item()
          correct += (answer_model == labels).sum().item()
          total += labels.size(0)

          if i % 100 == 0:
              print(f'Batch {i}/{len(val_loader)} is DONE!')

    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    return accuracy, avg_loss

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [ ]:
model_name = "microsoft/deberta-v3-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)


print(next(model.parameters()).dtype)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

torch.float32


In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

train_set = qqp_preprocessed['train']
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=32, shuffle=True, num_workers=2, collate_fn=transformers.default_data_collator
)

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=32, shuffle=False, num_workers=2, collate_fn=transformers.default_data_collator
)

In [ ]:
bad_params = []

for name, p in model.named_parameters():
    if torch.isnan(p).any():
        bad_params.append((name, "nan"))
    elif torch.isinf(p).any():
        bad_params.append((name, "inf"))

print("bad params count:", len(bad_params))
print("first bad params:", bad_params[:10])

bad params count: 0
first bad params: []


In [ ]:
for batch in train_loader:
    break
{k: v for k, v in batch.items()}

{'labels': tensor([0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0,
         0, 0, 0, 1, 0, 0, 0, 0]),
 'idx': tensor([114080, 262488, 281811, 145215, 227098, 249710, 105499,    759, 254758,
         346790, 208494, 320902, 274749, 240496,  44476, 172317, 204206,  58873,
         233689, 178293, 142491, 113396, 215439, 329022, 310892, 210011, 285100,
         115178,  57575,  82517, 154193, 230616]),
 'input_ids': tensor([[   1,  458,  269,  ...,    0,    0,    0],
         [   1, 3432,  273,  ...,    0,    0,    0],
         [   1,  273,  286,  ...,    0,    0,    0],
         ...,
         [   1, 1167,  269,  ...,    0,    0,    0],
         [   1,  273, 8796,  ...,    0,    0,    0],
         [   1,  458,  403,  ...,    0,    0,    0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0, 

In [ ]:
from tqdm.auto import tqdm
import torch

epochs = 1 # на самом деле модель обучалась на 2х эпохах, просто после первой у меня слетел код, потому что я не инициализировал evaluate
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")

    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"].long(),
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    val_acc, val_loss = evaluate(model, val_loader, device)

    print(f"\nEpoch {epoch + 1}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss:   {val_loss:.4f}")
    print(f"Val acc:    {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc

Epoch 1/1:   0%|          | 0/11371 [00:00<?, ?it/s]

Batch 0/1264 is DONE!
Batch 100/1264 is DONE!
Batch 200/1264 is DONE!
Batch 300/1264 is DONE!
Batch 400/1264 is DONE!
Batch 500/1264 is DONE!
Batch 600/1264 is DONE!
Batch 700/1264 is DONE!
Batch 800/1264 is DONE!
Batch 900/1264 is DONE!
Batch 1000/1264 is DONE!
Batch 1100/1264 is DONE!
Batch 1200/1264 is DONE!

Epoch 1
Train loss: 0.1906
Val loss:   0.2394
Val acc:    0.9116


In [1]:
import os
import torch

# сохраним модель
def save_checkpoint(model, tokenizer, save_dir, optimizer=None, epoch=None, step=None, extra_state=None):
    os.makedirs(save_dir, exist_ok=True)

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    training_state = {
        "epoch": epoch,
        "step": step,
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "extra_state": extra_state,
    }

    torch.save(training_state, os.path.join(save_dir, "training_state.pt"))
    print(f"Checkpoint сохранён в: {save_dir}")


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_checkpoint(model, tokenizer, "/content/drive/MyDrive/qqp_checkpoint", optimizer=optimizer, epoch=2)

### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [4]:
model_dir = "/content/drive/MyDrive/qqp_checkpoint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = transformers.AutoTokenizer.from_pretrained(model_dir)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_dir)
model = model.to(device).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print(next(model.parameters()).dtype)

Loading weights:   0%|          | 0/106 [00:01<?, ?it/s]

torch.float32


In [5]:
model.eval()

result = tokenizer(
    "okey, i'm find my key", "omg, i'm lost my key",
    padding='max_length', max_length=128, truncation=True, return_tensors="pt"
)

result = {k: v.to(device) for k, v in result.items()}

with torch.no_grad():
    outputs = model(**result)

scores = torch.softmax(outputs.logits, dim=-1)[:, 1]
print(scores.cpu().tolist())


[0.7543591260910034]


In [6]:
import torch
import heapq

def score_pairs(question_a, questions_b, model, tokenizer, device, max_length=128):
    model.eval()

    encoded = tokenizer(
        [question_a] * len(questions_b),
        questions_b,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        scores = torch.softmax(outputs.logits, dim=-1)[:, 1]

    return scores.cpu().tolist()


def find_topk_dupl(question_a, all_questions, model, tokenizer, device, top_k=5, batch_size=64, max_length=128):
    top_candidates = []

    for i in range(0, len(all_questions), batch_size):
        batch_questions = all_questions[i:i + batch_size]
        batch_scores = score_pairs(
            question_a,
            batch_questions,
            model,
            tokenizer,
            device,
            max_length=max_length
        )

        for q, s in zip(batch_questions, batch_scores):
            if len(top_candidates) < top_k:
                heapq.heappush(top_candidates, (s, q))
            else:
                heapq.heappushpop(top_candidates, (s, q))

    top_candidates = sorted(top_candidates, key=lambda x: x[0], reverse=True)

    return [(q, s) for s, q in top_candidates]



In [20]:
test_questions = [
    "How can I improve my English speaking skills?",
    "What is the best way to lose weight fast?",
    "How do I earn money online?",
    "What are the symptoms of depression?",
    "How can I learn Python programming?"
]

# в данном случае я взял 5000 примеров, потому что если брать больше, то слетает окружение колаба
# за счет этого потерялось качество и есть предложения с очень низким score
sample_questions = list(qqp['train']['text1'][:5000])

for i, question in enumerate(test_questions, 1):
    print("=" * 80)
    print(f"Example {i}")
    print(f"Input question: {question}\n")

    results = find_topk_dupl(
        question_a=question,
        all_questions=sample_questions,
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=5,
        batch_size=16
    )

    print("Top-5 potential duplicates:")
    for rank, (candidate, score) in enumerate(results, 1):
        print(f"{rank}. score={score:.4f} | {candidate}")
    print()

Example 1
Input question: How can I improve my English speaking skills?

Top-5 potential duplicates:
1. score=0.9997 | How can I improve my english language skills? I am basically from gujarati background.
2. score=0.9997 | How can I improve my english language skills? I am basically from gujarati background.
3. score=0.9994 | How can I increase my English fluency?
4. score=0.9994 | How I can speak English with fluency?
5. score=0.9991 | What should I do to improve my English ?

Example 2
Input question: What is the best way to lose weight fast?

Top-5 potential duplicates:
1. score=0.9997 | How can I lose 10 Kilos?
2. score=0.9996 | How can I lose 4kg weight?
3. score=0.9994 | What is the best way to be in a calorie deficit and lose weight successfully?
4. score=0.9993 | How do I lose fats and excessive weight from body?
5. score=0.9985 | How could I lose a few pounds quickly?

Example 3
Input question: How do I earn money online?

Top-5 potential duplicates:
1. score=0.9991 | How do 

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# чтобы улучшить алгоритм, мы можем сделать гибридный подход к задача
# в данном случае я решил использовать TF-IDF, который на первом этапе отберет 100 близких примеров и уже после на этих 100 примерах будем использовать модель
# во-первых, это дает возможность работать со всем датасетом
# во-вторых, вся система работает в разы быстрее, так как в модель идет только валидная часть, а не весь датасет
all_questions = list(qqp['train']['text1'])

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(all_questions)
X.shape

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

test_questions = [
    "How can I improve my English speaking skills?",
    "What is the best way to lose weight fast?",
    "How do I earn money online?",
    "What are the symptoms of depression?",
    "How can I learn Python programming?"
]
top_k = 100

for i, question in enumerate(test_questions, 1):
    print("=" * 80)
    print(f"Example {i}")
    print(f"Input question: {question}\n")

    question_tf_idf = vectorizer.transform([question])

    sims = cosine_similarity(question_tf_idf, X).flatten()

    top_idx = np.argsort(sims)[-top_k:][::-1]

    sample_questions = [all_questions[idx] for idx in top_idx]
    sample_questions = list(dict.fromkeys(sample_questions))

    results = find_topk_dupl(
        question_a=question,
        all_questions=sample_questions,
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=5,
        batch_size=16
    )

    print("Top-5 potential duplicates:")
    for rank, (candidate, score) in enumerate(results, 1):
        print(f"{rank}. score={score:.4f} | {candidate}")
    print()


Example 1
Input question: How can I improve my English speaking skills?

Top-5 potential duplicates:
1. score=0.9983 | How can improve my English speaking?
2. score=0.9982 | How do I can improve English speaking?
3. score=0.9982 | How do I improve my English speaking?
4. score=0.9981 | What can I do to improve my English speaking?
5. score=0.9975 | How can I improve my English skills?

Example 2
Input question: What is the best way to lose weight fast?

Top-5 potential duplicates:
1. score=0.9983 | What is the best way to reduce weight fast?
2. score=0.9982 | What are the best ways to lose weight fast?
3. score=0.9967 | What's the best diet to lose weight fast?
4. score=0.9965 | What is the best and quick way to lose weight?
5. score=0.9965 | How can I lose my weight fast?

Example 3
Input question: How do I earn money online?

Top-5 potential duplicates:
1. score=0.9952 | How can I earn money online?
2. score=0.9951 | What should I do to earn money online?
3. score=0.9949 | Can I earn

Как мы видим, алгоритм отработал лучше